# Market Risk Watcher — Rebuilt Notebook

This notebook rebuilds the core workflow:

1. Download Yahoo Finance data
2. Connect to PostgreSQL
3. Build and load `assets`
4. Transform and load `prices`
5. Create the `asset_risk_summary` view
6. Read bond-risk results back into Python

**Important:** the insert cells are guarded so they do not reinsert data if the tables are already populated.


In [1]:
%pip install -q yfinance pandas sqlalchemy psycopg2-binary python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

import yfinance as yf
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [3]:
stock_tickers = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "AMD", "INTC", "JPM",
    "BAC", "GS", "JNJ", "PFE", "XOM",
    "CVX", "WMT", "KO", "DIS", "NKE"
]

equity_etfs = [
    "SPY", "QQQ", "IWM", "VTI"
]

bond_etfs = [
    "BND", "TLT", "IEF", "HYG", "LQD", "BGRN"
]

other_etfs = [
    "GLD", "USO", "VNQ", "EEM", "VGK"
]

indices = [
    "^GSPC", "^IXIC", "^DJI", "^VIX", "^TNX"
]

tickers = (
    stock_tickers
    + equity_etfs
    + bond_etfs
    + other_etfs
    + indices
)

print("Ticker count:", len(tickers))

Ticker count: 40


In [4]:
data = yf.download(
    tickers,
    start="2020-01-01",
    end="2026-08-24",
    auto_adjust=False,
    group_by="ticker"
)

print("Downloaded shape:", data.shape)

[*********************100%***********************]  40 of 40 completed

Downloaded shape: (1669, 240)


In [5]:
display(data.head())
print(data.columns)

Ticker       ^VIX                                                 AMZN  \
Price        Open       High    Low  Close Adj Close Volume       Open   
Date                                                                     
2020-01-02  13.46  13.720000  12.42  12.47     12.47      0  93.750000   
2020-01-03  15.01  16.200001  13.13  14.02     14.02      0  93.224998   
2020-01-06  15.45  16.389999  13.54  13.85     13.85      0  93.000000   
2020-01-07  13.84  14.460000  13.39  13.79     13.79      0  95.224998   
2020-01-08  15.16  15.240000  12.83  13.45     13.45      0  94.902000   

Ticker                                       ...         GLD              \
Price            High        Low      Close  ...         Low       Close   
Date                                         ...                           
2020-01-02  94.900497  93.207497  94.900497  ...  143.399994  143.949997   
2020-01-03  94.309998  93.224998  93.748497  ...  145.399994  145.860001   
2020-01-06  95.184502  93.000000  95.143997  ...  146.949997  147.389999   
2020-01-07  95.694504  94.601997  95.343002  ...  147.429993  147.970001   
2020-01-08  95.550003  94.321999  94.598503  ...  146.139999  146.860001   

Ticker                                     KO                        \
Price        Adj Close      Volume       Open       High        Low   
Date                                                                  
2020-01-02  143.949997   7733800.0  55.320000  55.430000  54.759998   
2020-01-03  145.860001  12272800.0  54.320000  54.990002  54.090000   
2020-01-06  147.389999  14403300.0  54.650002  54.910000  54.520000   
2020-01-07  147.970001   7978500.0  54.450001  54.599998  54.150002   
2020-01-08  146.860001  22248500.0  54.270000  54.639999  54.150002   

Ticker                                        
Price           Close  Adj Close      Volume  
Date                                          
2020-01-02  54.990002  45.141270  11867700.0  
2020-01-03  54.689999  44.895000  11354500.0  
2020-01-06  54.669998  44.878563  14698300.0  
2020-01-07  54.250000  44.533791   9973900.0  
2020-01-08  54.349998  44.615883  10676000.0  

[5 rows x 240 columns]

MultiIndex([('^VIX',      'Open'),
            ('^VIX',      'High'),
            ('^VIX',       'Low'),
            ('^VIX',     'Close'),
            ('^VIX', 'Adj Close'),
            ('^VIX',    'Volume'),
            ('AMZN',      'Open'),
            ('AMZN',      'High'),
            ('AMZN',       'Low'),
            ('AMZN',     'Close'),
            ...
            ( 'GLD',       'Low'),
            ( 'GLD',     'Close'),
            ( 'GLD', 'Adj Close'),
            ( 'GLD',    'Volume'),
            (  'KO',      'Open'),
            (  'KO',      'High'),
            (  'KO',       'Low'),
            (  'KO',     'Close'),
            (  'KO', 'Adj Close'),
            (  'KO',    'Volume')],
           names=['Ticker', 'Price'], length=240)


In [6]:
print("Notebook working directory:")
print(Path.cwd())

print("\n.env exists:")
print(Path(".env").exists())

Notebook working directory:
/Users/kdog/Desktop/Yeni Proje/Market-Risk-Watcher

.env exists:
True


In [7]:
load_dotenv()

database_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(
    database_url,
    pool_pre_ping=True
)

with engine.connect() as connection:
    database = connection.execute(
        text("SELECT current_database();")
    ).scalar()

print("Connected to:", database)

Connected to: market_risk


In [8]:
assets_data = []

for ticker in tickers:
    if ticker in stock_tickers:
        asset_type = "stock"
    elif ticker in equity_etfs:
        asset_type = "equity_etf"
    elif ticker in bond_etfs:
        asset_type = "bond_etf"
    elif ticker in other_etfs:
        asset_type = "other_etf"
    elif ticker in indices:
        asset_type = "index"
    else:
        asset_type = "unknown"

    assets_data.append(
        {
            "ticker": ticker,
            "asset_type": asset_type
        }
    )

assets_df = pd.DataFrame(assets_data)

display(assets_df.head())
print("Shape:", assets_df.shape)

,ticker,asset_type
0,AAPL,stock
1,MSFT,stock
2,NVDA,stock
3,AMZN,stock
4,GOOGL,stock


Shape: (40, 2)


In [9]:
def get_asset_name(ticker):
    try:
        info = yf.Ticker(ticker).get_info()

        return (
            info.get("longName")
            or info.get("shortName")
            or ticker
        )
    except Exception as exc:
        print(f"Could not get name for {ticker}: {exc}")
        return ticker


assets_df["company_name"] = assets_df["ticker"].apply(
    get_asset_name
)

assets_df = assets_df[
    ["ticker", "company_name", "asset_type"]
]

display(assets_df.head(10))

,ticker,company_name,asset_type
0,AAPL,Apple Inc.,stock
1,MSFT,Microsoft Corporation,stock
2,NVDA,NVIDIA Corporation,stock
3,AMZN,"Amazon.com, Inc.",stock
4,GOOGL,Alphabet Inc.,stock
5,META,"Meta Platforms, Inc.",stock
6,TSLA,"Tesla, Inc.",stock
7,AMD,"Advanced Micro Devices, Inc.",stock
8,INTC,Intel Corporation,stock
9,JPM,JPMorgan Chase & Co.,stock


In [10]:
with engine.connect() as connection:
    assets_count = connection.execute(
        text("SELECT COUNT(*) FROM assets;")
    ).scalar()

print("Existing assets rows:", assets_count)

if assets_count == 0:
    inserted = assets_df.to_sql(
        name="assets",
        con=engine,
        schema="public",
        if_exists="append",
        index=False
    )
    print("Inserted asset rows:", inserted)
else:
    print("Assets table is already populated, so insert was skipped.")

Existing assets rows: 40
Assets table is already populated, so insert was skipped.


In [11]:
asset_lookup = pd.read_sql(
    """
    SELECT
        asset_id,
        ticker
    FROM assets
    ORDER BY asset_id;
    """,
    engine
)

ticker_to_id = dict(
    zip(
        asset_lookup["ticker"],
        asset_lookup["asset_id"]
    )
)

display(asset_lookup.head())
print("Mapped tickers:", len(ticker_to_id))

,asset_id,ticker
0,1,AAPL
1,2,MSFT
2,3,NVDA
3,4,AMZN
4,5,GOOGL


Mapped tickers: 40


In [12]:
price_frames = []

for ticker in tickers:
    ticker_df = data[ticker].copy()
    ticker_df = ticker_df.reset_index()

    ticker_df["ticker"] = ticker

    ticker_df = ticker_df[
        [
            "ticker",
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume"
        ]
    ]

    price_frames.append(ticker_df)

prices_df = pd.concat(
    price_frames,
    ignore_index=True
)

prices_df = prices_df.rename(
    columns={
        "Date": "date",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume"
    }
)

display(prices_df.head())
print("Rows before cleaning:", len(prices_df))

Price,ticker,date,open,high,low,close,volume
0,AAPL,2020-01-02,74.059998,75.150002,73.797501,75.087502,135480400.0
1,AAPL,2020-01-03,74.287498,75.144997,74.125000,74.357498,146322800.0
2,AAPL,2020-01-06,73.447502,74.989998,73.187500,74.949997,118387200.0
3,AAPL,2020-01-07,74.959999,75.224998,74.370003,74.597504,108872000.0
4,AAPL,2020-01-08,74.290001,76.110001,74.290001,75.797501,132079200.0


Rows before cleaning: 66760


In [13]:
prices_df["asset_id"] = prices_df["ticker"].map(
    ticker_to_id
)

print(
    "Missing asset IDs:",
    prices_df["asset_id"].isna().sum()
)

prices_df = prices_df.dropna(
    subset=[
        "open",
        "high",
        "low",
        "close"
    ]
)

prices_df["date"] = pd.to_datetime(
    prices_df["date"]
).dt.date

prices_df["volume"] = pd.to_numeric(
    prices_df["volume"],
    errors="coerce"
).astype("Int64")

prices_df = prices_df[
    [
        "asset_id",
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
]

display(prices_df.head())
print("Clean price rows:", len(prices_df))
print()
print(prices_df.dtypes)

Missing asset IDs: 0


Price,asset_id,date,open,high,low,close,volume
0,1,2020-01-02,74.059998,75.150002,73.797501,75.087502,135480400
1,1,2020-01-03,74.287498,75.144997,74.125000,74.357498,146322800
2,1,2020-01-06,73.447502,74.989998,73.187500,74.949997,118387200
3,1,2020-01-07,74.959999,75.224998,74.370003,74.597504,108872000
4,1,2020-01-08,74.290001,76.110001,74.290001,75.797501,132079200


Clean price rows: 66721

Price
asset_id      int64
date         object
open        float64
high        float64
low         float64
close       float64
volume        Int64
dtype: object


In [14]:
with engine.connect() as connection:
    prices_count = connection.execute(
        text("SELECT COUNT(*) FROM prices;")
    ).scalar()

print("Existing price rows:", prices_count)

if prices_count == 0:
    inserted = prices_df.to_sql(
        name="prices",
        con=engine,
        schema="public",
        if_exists="append",
        index=False,
        chunksize=5000
    )
    print("Inserted price rows:", inserted)
else:
    print("Prices table is already populated, so insert was skipped.")

Existing price rows: 66721
Prices table is already populated, so insert was skipped.


In [15]:
with engine.connect() as connection:
    asset_count = connection.execute(
        text("SELECT COUNT(*) FROM assets;")
    ).scalar()

    price_count = connection.execute(
        text("SELECT COUNT(*) FROM prices;")
    ).scalar()

print("Assets in SQL:", asset_count)
print("Prices in SQL:", price_count)

Assets in SQL: 40
Prices in SQL: 66721


In [16]:
create_view_sql = text(
    """
    CREATE OR REPLACE VIEW asset_risk_summary AS

    WITH calculations AS (
        SELECT
            p.asset_id,
            a.ticker,
            a.company_name,
            a.asset_type,
            p.date,
            p.close,
            (
                p.close /
                LAG(p.close) OVER (
                    PARTITION BY p.asset_id
                    ORDER BY p.date
                )
                - 1
            ) AS daily_return,
            AVG(p.close) OVER (
                PARTITION BY p.asset_id
                ORDER BY p.date
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS moving_avg_20d
        FROM prices p
        JOIN assets a
            ON p.asset_id = a.asset_id
    ),

    risk_metrics AS (
        SELECT
            asset_id,
            ticker,
            company_name,
            asset_type,
            date,
            close,
            daily_return,
            moving_avg_20d,
            STDDEV_SAMP(daily_return) OVER (
                PARTITION BY asset_id
                ORDER BY date
                ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
            ) AS volatility_20d,
            STDDEV_SAMP(daily_return) OVER (
                PARTITION BY asset_id
                ORDER BY date
                ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
            ) AS volatility_60d
        FROM calculations
    ),

    latest_rows AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY asset_id
                ORDER BY date DESC
            ) AS row_number
        FROM risk_metrics
    )

    SELECT
        asset_id,
        ticker,
        company_name,
        asset_type,
        date,
        close,
        ROUND(
            daily_return * 100,
            2
        ) AS daily_return_pct,
        ROUND(
            moving_avg_20d,
            2
        ) AS moving_avg_20d,
        ROUND(
            volatility_20d * SQRT(252::numeric) * 100,
            2
        ) AS volatility_20d_pct,
        ROUND(
            volatility_60d * SQRT(252::numeric) * 100,
            2
        ) AS volatility_60d_pct
    FROM latest_rows
    WHERE row_number = 1;
    """
)

with engine.begin() as connection:
    connection.execute(create_view_sql)

print("asset_risk_summary view is ready.")

asset_risk_summary view is ready.


In [17]:
bond_risk_df = pd.read_sql(
    """
    SELECT
        ticker,
        company_name,
        date,
        close,
        daily_return_pct,
        moving_avg_20d,
        volatility_20d_pct,
        volatility_60d_pct,
        CASE
            WHEN volatility_20d_pct < 5
                THEN 'LOW'
            WHEN volatility_20d_pct < 10
                THEN 'MEDIUM'
            ELSE 'HIGH'
        END AS risk_level
    FROM asset_risk_summary
    WHERE asset_type = 'bond_etf'
    ORDER BY volatility_20d_pct DESC;
    """,
    engine
)

display(bond_risk_df)

,ticker,company_name,date,close,daily_return_pct,moving_avg_20d,volatility_20d_pct,volatility_60d_pct,risk_level
0,TLT,iShares 20+ Year Treasury Bond ETF,2026-08-21,82.050003,-0.35,82.53,11.96,9.70,HIGH
1,LQD,iShares iBoxx $ Investment Grade Corporate Bon...,2026-08-21,105.919998,-0.13,106.28,5.91,5.39,MEDIUM
2,IEF,iShares 7-10 Year Treasury Bond ETF,2026-08-21,92.820000,-0.19,93.08,4.86,4.96,LOW
3,BND,Vanguard Total Bond Market Index Fund ETF Shares,2026-08-21,72.230003,-0.15,72.34,4.09,3.97,LOW
4,BGRN,iShares USD Green Bond ETF,2026-08-21,46.884998,-0.02,46.89,3.12,3.17,LOW
5,HYG,iShares iBoxx $ High Yield Corporate Bond ETF,2026-08-21,79.610001,0.06,79.52,2.74,3.42,LOW


In [18]:
all_risk_df = pd.read_sql(
    """
    SELECT
        *
    FROM asset_risk_summary
    ORDER BY volatility_20d_pct DESC;
    """,
    engine
)

display(all_risk_df.head(10))

,asset_id,ticker,company_name,asset_type,date,close,daily_return_pct,moving_avg_20d,volatility_20d_pct,volatility_60d_pct
0,39,^VIX,CBOE Volatility Index,index,2026-08-21,15.130000,-5.50,16.00,102.91,137.36
1,8,AMD,"Advanced Micro Devices, Inc.",stock,2026-08-21,473.250000,0.81,481.12,80.56,77.31
2,9,INTC,Intel Corporation,stock,2026-08-21,90.070000,-2.24,95.70,74.89,84.72
3,4,AMZN,"Amazon.com, Inc.",stock,2026-08-21,258.630005,-0.57,261.38,62.95,45.17
4,32,USO,"United States Oil Fund, LP",other_etf,2026-08-21,134.639999,0.07,125.72,61.31,52.63
5,2,MSFT,Microsoft Corporation,stock,2026-08-21,483.239990,0.43,473.09,60.37,47.47
6,6,META,"Meta Platforms, Inc.",stock,2026-08-21,549.900024,0.75,576.48,46.59,48.43
7,5,GOOGL,Alphabet Inc.,stock,2026-08-21,344.820007,1.22,348.40,39.28,37.98
8,3,NVDA,NVIDIA Corporation,stock,2026-08-21,214.720001,-0.98,213.18,38.14,40.11
9,17,WMT,Walmart Inc.,stock,2026-08-21,103.699997,-0.13,112.21,37.66,29.04
